In [ ]:


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------

import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
ANALYTICS_DIR = SCRIPT_DIR.parent

if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))

from utils.export_utils import exportar_csv

PROJECT_ROOT = Path(__file__).resolve().parents[3]
OUTPUT_DIR = PROJECT_ROOT / "scripts" / "Analytics" / "outputs" / "gold_05"


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics_Gold05")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# GOLD 05 - Qual é o índice de adoção de Inteligência Artificial e seu impacto?
# ---------------------------------------------------------------------

caminho_gold_05 = PROJECT_ROOT / "Gold" / "perguntas_negocio" / "gold_05_adocao_ia"

arquivos_gold_05 = [
    str(arquivo) for arquivo in caminho_gold_05.glob("part-*.csv")
]

df_ia = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_05)
)

if "variavel_original" in df_ia.columns:
    df_ia = df_ia.withColumnRenamed("variavel_original", "variavel")


# ---------------------------------------------------------------------
# ÍNDICE DE ADOÇÃO PESSOAL DE IA
"""
O índice de adoção pessoal é derivado da opção de não uso. Como essa resposta identifica diretamente quem declarou não utilizar soluções de IA generativa para produtividade, a adoção é calculada como o complemento desse percentual dentro do total de elegíveis.
"""
# ---------------------------------------------------------------------

df_nao_uso = (
    df_ia.filter(
        (F.col("categoria") == "uso_pessoal_chatgpt_copilot")
        & (
            F.col("opcao")
            == "nao_uso_solucoes_de_ai_generativa_com_foco_em_produtividade"
        )
    )
)

indice_adocao_pessoal = (
    df_nao_uso
    .select(
        "edicao",
        "elegiveis",
        F.col("selecionaram").cast("int").alias("nao_usam_ia"),
        (F.col("elegiveis") - F.col("selecionaram")).cast("int").alias("usam_ia"),
        F.round(100 - F.col("pct_adocao"), 1).alias("pct_adocao_pessoal")
    )
    .orderBy("edicao")
)

print("\n" + "=" * 100)
print("01. ÍNDICE DE ADOÇÃO PESSOAL DE IA")
print("=" * 100)

indice_adocao_pessoal.show(truncate=False)
"""
Nos outputs, a adoção pessoal cresce de 80,3% em 2023-2024 para 93,5% em 2024-2025 e 97,9% em 2025-2026. No mesmo período, o número de pessoas que declararam não usar IA cai de 744 para 44.
"""


# ---------------------------------------------------------------------
# EVOLUÇÃO DO ÍNDICE DE ADOÇÃO PESSOAL
"""
As três edições são colocadas lado a lado para medir a mudança em pontos percentuais entre o início e o fim da série. Esse formato evita interpretar a diferença como crescimento relativo e mantém a leitura na mesma unidade do indicador.
"""
# ---------------------------------------------------------------------

comparativo_adocao_pessoal = (
    indice_adocao_pessoal
    .select("edicao", "pct_adocao_pessoal")
    .groupBy()
    .pivot(
        "edicao",
        ["2023-2024", "2024-2025", "2025-2026"]
    )
    .agg(F.first("pct_adocao_pessoal"))
    .withColumn(
        "variacao_pp_23_26",
        F.round(F.col("2025-2026") - F.col("2023-2024"), 1)
    )
)

print("\n" + "=" * 100)
print("02. EVOLUÇÃO DA ADOÇÃO PESSOAL DE IA")
print("=" * 100)

comparativo_adocao_pessoal.show(truncate=False)
"""
Entre 2023-2024 e 2025-2026, o índice de adoção pessoal aumenta 17,6 pontos percentuais, passando de 80,3% para 97,9%.
"""


# ---------------------------------------------------------------------
# PRIORIDADE DA IA NAS EMPRESAS - DISPONÍVEL A PARTIR DE 2024-2025
"""
A análise de prioridade corporativa começa em 2024-2025 porque esse indicador não está disponível na edição 2023-2024. As respostas originais são resumidas em cinco níveis ordenados para facilitar a comparação entre as duas edições disponíveis.
"""
# ---------------------------------------------------------------------

df_prioridade = (
    df_ia
    .filter(F.col("categoria") == "ia_generativa_prioridade_na_empresa")
    .withColumn(
        "prioridade",
        F.when(
            F.col("valor").contains("principal prioridade como empresa"),
            "Principal prioridade"
        )
        .when(
            F.col("valor").contains("principais prioridades para os próximos"),
            "Entre as principais prioridades"
        )
        .when(
            F.col("valor").startswith("Mais ou menos"),
            "Iniciativa secundária"
        )
        .when(
            F.col("valor").startswith("Não é uma iniciativa"),
            "Não é prioridade"
        )
        .when(
            F.col("valor").startswith("Não sei"),
            "Não sabe opinar"
        )
        .otherwise(F.col("valor"))
    )
    .withColumn(
        "ordem",
        F.when(F.col("prioridade") == "Principal prioridade", 1)
        .when(F.col("prioridade") == "Entre as principais prioridades", 2)
        .when(F.col("prioridade") == "Iniciativa secundária", 3)
        .when(F.col("prioridade") == "Não é prioridade", 4)
        .otherwise(5)
    )
    .select(
        "edicao",
        "prioridade",
        "contagem",
        "total_respondentes",
        "pct_na_dimensao",
        "ordem"
    )
    .orderBy("edicao", "ordem")
)

print("\n" + "=" * 100)
print("03. PRIORIDADE DA IA NAS EMPRESAS")
print("=" * 100)

df_prioridade.show(truncate=False)
"""
Os outputs mostram aumento da percepção de IA como tema prioritário: a categoria "Entre as principais prioridades" passa de 31,1% para 36,8%, enquanto "Não é prioridade" recua de 14,8% para 11,3%.
"""


# ---------------------------------------------------------------------
# RESPONDENTES QUE INDICAM IA COMO PRIORIDADE ALTA NA EMPRESA - PRINCIPAL PRIORIDADE + ENTRE AS PRINCIPAIS PRIORIDADES
"""
Para sintetizar prioridade alta, são agrupadas apenas as duas respostas mais fortes da escala: "Principal prioridade" e "Entre as principais prioridades". O agrupamento preserva o total de respondentes de cada edição como denominador.
"""
# ---------------------------------------------------------------------

prioridade_alta = (
    df_prioridade
    .groupBy("edicao")
    .agg(
        F.sum(
            F.when(
                F.col("ordem").isin(1, 2),
                F.col("contagem")
            ).otherwise(0)
        ).alias("respondentes_prioridade_alta"),
        F.max("total_respondentes").alias("total_respondentes")
    )
    .withColumn(
        "pct_prioridade_alta",
        F.round(
            F.col("respondentes_prioridade_alta")
            / F.col("total_respondentes")
            * 100,
            1
        )
    )
    .orderBy("edicao")
)

print("\n" + "=" * 100)
print("04. RESPONDENTES QUE INDICAM IA COMO PRIORIDADE ALTA NA EMPRESA")
print("=" * 100)

prioridade_alta.show(truncate=False)
"""
A parcela de respondentes que classifica IA como prioridade alta sobe de 53,6% em 2024-2025 para 60,6% em 2025-2026, uma diferença de 7,0 pontos percentuais.
"""


# ---------------------------------------------------------------------
# FORMAS DE USO DE IA NAS EMPRESAS
"""
As opções de uso corporativo são comparadas individualmente porque representam formas de adoção que podem coexistir na mesma empresa. Por isso, os percentuais não devem ser interpretados como partes mutuamente exclusivas de uma distribuição.
"""
# ---------------------------------------------------------------------

df_uso_empresa = (
    df_ia
    .filter(F.col("categoria") == "tipo_de_uso_de_ia_na_empresa")
    .withColumn(
        "tipo_uso",
        F.when(
            F.col("opcao")
            == "colaboradores_usando_ai_generativa_de_forma_independente_e_descentralizada",
            "Uso independente/descentralizado"
        )
        .when(
            F.col("opcao")
            == "direcionamento_centralizado_do_uso_de_ai_generativa",
            "Direcionamento centralizado"
        )
        .when(
            F.col("opcao") == "desenvolvedores_utilizando_copilots",
            "Desenvolvedores usando Copilots"
        )
        .when(
            F.col("opcao")
            == "ai_generativa_e_llms_para_melhorar_produtos_internos_para_os_colaboradores",
            "Melhoria de produtos internos"
        )
        .when(
            F.col("opcao")
            == "ai_generativa_e_llms_para_melhorar_produtos_externos_para_os_clientes_finais",
            "Melhoria de produtos externos"
        )
        .when(
            F.col("opcao")
            == "ia_generativa_e_llms_como_principal_frente_do_negocio",
            "IA como principal frente do negócio"
        )
        .when(
            F.col("opcao") == "ia_generativa_e_llms_nao_e_prioridade",
            "IA não é prioridade"
        )
        .when(
            F.col("opcao")
            == "nao_sei_opinar_sobre_o_uso_de_ia_generativa_e_llms_na_empresa",
            "Não sabe opinar"
        )
        .otherwise(F.col("opcao"))
    )
)

comparativo_uso_empresa = (
    df_uso_empresa
    .groupBy("tipo_uso")
    .pivot(
        "edicao",
        ["2023-2024", "2024-2025", "2025-2026"]
    )
    .agg(F.first("pct_adocao"))
    .withColumn(
        "variacao_pp_23_26",
        F.round(F.col("2025-2026") - F.col("2023-2024"), 1)
    )
    .orderBy(F.desc("2025-2026"))
)

print("\n" + "=" * 100)
print("05. EVOLUÇÃO DO USO DE IA NAS EMPRESAS")
print("=" * 100)

comparativo_uso_empresa.show(truncate=False)
"""
O maior avanço ocorre no direcionamento centralizado, que passa de 10,9% para 36,3% (+25,4 p.p.). Desenvolvedores usando Copilots também cresce 18,4 p.p., enquanto a indicação de que IA não é prioridade cai 14,2 p.p. Apesar da maior formalização, o uso independente/descentralizado continua sendo a forma mais frequente em 2025-2026, com 48,8%.
"""


# ---------------------------------------------------------------------
# USO PESSOAL DE SOLUÇÕES DE IA
"""
O histórico de uso pessoal detalha como o acesso às soluções mudou, complementando o índice geral de adoção. Como uma pessoa pode utilizar mais de uma modalidade, cada opção é analisada pela sua própria taxa de adoção.
"""
# ---------------------------------------------------------------------

df_uso_pessoal = (
    df_ia
    .filter(F.col("categoria") == "uso_pessoal_chatgpt_copilot")
    .withColumn(
        "tipo_uso",
        F.when(
            F.col("opcao")
            == "a_empresa_que_trabalho_paga_pelas_solucoes_de_ai_generativa_com_foco_em_produtividade",
            "Empresa paga pela solução"
        )
        .when(
            F.col("opcao")
            == "uso_solucoes_gratuitas_de_ai_generativa_com_foco_em_produtividade",
            "Uso soluções gratuitas"
        )
        .when(
            F.col("opcao") == "uso_solucoes_do_tipo_copilot",
            "Uso soluções tipo Copilot"
        )
        .when(
            F.col("opcao")
            == "uso_e_pago_pelas_solucoes_de_ai_generativa_com_foco_em_produtividade",
            "Uso e pago pela solução"
        )
        .when(
            F.col("opcao")
            == "nao_uso_solucoes_de_ai_generativa_com_foco_em_produtividade",
            "Não uso IA generativa"
        )
        .otherwise(F.col("opcao"))
    )
)

comparativo_uso_pessoal = (
    df_uso_pessoal
    .groupBy("tipo_uso")
    .pivot(
        "edicao",
        ["2023-2024", "2024-2025", "2025-2026"]
    )
    .agg(F.first("pct_adocao"))
    .withColumn(
        "variacao_pp_23_26",
        F.round(F.col("2025-2026") - F.col("2023-2024"), 1)
    )
    .orderBy(F.desc("2025-2026"))
)

print("\n" + "=" * 100)
print("06. EVOLUÇÃO DO USO PESSOAL DE IA")
print("=" * 100)

comparativo_uso_pessoal.show(truncate=False)
"""
Os outputs indicam mudança no modelo de acesso: soluções pagas pela empresa avançam de 6,4% para 42,4% (+36,0 p.p.), enquanto o uso de soluções gratuitas cai de 63,7% para 30,5% (-33,2 p.p.). O uso de Copilots cresce 18,0 p.p. e a declaração de não uso recua de 19,7% para 2,1%.
"""


# ---------------------------------------------------------------------
# BARREIRAS PARA ADOÇÃO DE IA
"""
As barreiras são comparadas pela taxa de seleção de cada motivo entre os elegíveis. Como mais de uma barreira pode ser indicada pelo mesmo respondente, os percentuais são avaliados individualmente e não devem ser somados como uma distribuição única.
"""
# ---------------------------------------------------------------------

df_barreiras = (
    df_ia
    .filter(F.col("categoria") == "motivos_para_nao_usar_ia")
    .withColumn(
        "barreira",
        F.when(
            F.col("opcao") == "falta_de_expertise_ou_falta_de_recursos",
            "Falta de expertise ou recursos"
        )
        .when(
            F.col("opcao")
            == "dados_da_empresa_nao_estao_prontos_para_uso_de_ia_generativa",
            "Dados não estão preparados"
        )
        .when(
            F.col("opcao") == "falta_de_compreensao_dos_casos_de_uso",
            "Falta de compreensão dos casos de uso"
        )
        .when(
            F.col("opcao")
            == "retorno_sobre_investimento_roi_nao_comprovado_de_ia_generativa",
            "ROI não comprovado"
        )
        .when(
            F.col("opcao") == "preocupacoes_com_seguranca_e_privacidade_de_dados",
            "Segurança e privacidade"
        )
        .when(
            F.col("opcao")
            == "falta_de_confiabilidade_das_saidas_alucinacao_dos_modelos",
            "Confiabilidade das respostas"
        )
        .when(
            F.col("opcao") == "incerteza_em_relacao_a_regulamentacao",
            "Incerteza regulatória"
        )
        .when(
            F.col("opcao") == "preocupacoes_com_propriedade_intelectual",
            "Propriedade intelectual"
        )
        .when(
            F.col("opcao")
            == "alta_direcao_da_empresa_nao_ve_valor_ou_nao_ve_como_prioridade",
            "Alta direção não vê valor/prioridade"
        )
        .otherwise(F.col("opcao"))
    )
)

comparativo_barreiras = (
    df_barreiras
    .groupBy("barreira")
    .pivot(
        "edicao",
        ["2023-2024", "2024-2025", "2025-2026"]
    )
    .agg(F.first("pct_adocao"))
    .withColumn(
        "variacao_pp_23_26",
        F.round(F.col("2025-2026") - F.col("2023-2024"), 1)
    )
    .orderBy(F.desc("2025-2026"))
)

print("\n" + "=" * 100)
print("07. EVOLUÇÃO DAS BARREIRAS PARA ADOÇÃO DE IA")
print("=" * 100)

comparativo_barreiras.show(truncate=False)
"""
A natureza das barreiras muda ao longo do período. Falta de expertise ou recursos chega a 38,8% em 2025-2026 e dados não preparados a 36,8%. O ROI não comprovado apresenta o maior crescimento histórico, de 15,3% para 28,4% (+13,1 p.p.), enquanto falta de compreensão dos casos de uso recua 4,8 p.p.
"""


# ---------------------------------------------------------------------
# TOP 5 BARREIRAS ATUAIS
"""
O recorte final prioriza as cinco barreiras com maior adoção na edição mais recente para destacar os obstáculos mais relevantes no cenário atual, sem misturar o ranking com variações históricas.
"""
# ---------------------------------------------------------------------

top_barreiras_atual = (
    df_barreiras
    .filter(F.col("edicao") == "2025-2026")
    .select(
        "barreira",
        "selecionaram",
        "elegiveis",
        "pct_adocao"
    )
    .orderBy(F.desc("pct_adocao"))
    .limit(5)
)

print("\n" + "=" * 100)
print("08. TOP 5 BARREIRAS PARA ADOÇÃO DE IA - 2025-2026")
print("=" * 100)

top_barreiras_atual.show(truncate=False)
"""
Em 2025-2026, as cinco principais barreiras são falta de expertise ou recursos (38,8%), dados não preparados (36,8%), falta de compreensão dos casos de uso (32,0%), ROI não comprovado (28,4%) e segurança e privacidade (27,8%).
"""


# ---------------------------------------------------------------------
# EXPORTAÇÃO DOS RESULTADOS PARA VISUALIZAÇÃO
# ---------------------------------------------------------------------

exportar_csv(
    indice_adocao_pessoal,
    OUTPUT_DIR,
    "ia_adocao_pessoal_historico.csv"
)

exportar_csv(
    comparativo_adocao_pessoal,
    OUTPUT_DIR,
    "ia_adocao_pessoal_comparativo.csv"
)

exportar_csv(
    df_prioridade,
    OUTPUT_DIR,
    "ia_prioridade_empresa.csv"
)

exportar_csv(
    prioridade_alta,
    OUTPUT_DIR,
    "ia_prioridade_alta.csv"
)

exportar_csv(
    comparativo_uso_empresa,
    OUTPUT_DIR,
    "ia_uso_empresa_historico.csv"
)

exportar_csv(
    comparativo_uso_pessoal,
    OUTPUT_DIR,
    "ia_uso_pessoal_historico.csv"
)

exportar_csv(
    comparativo_barreiras,
    OUTPUT_DIR,
    "ia_barreiras_historico.csv"
)

exportar_csv(
    top_barreiras_atual,
    OUTPUT_DIR,
    "ia_top_5_barreiras_2025_2026.csv"
)

print("\n" + "=" * 100)
print("EXPORTAÇÃO CONCLUÍDA")
print("=" * 100)

print("Outputs salvos em:")
print(OUTPUT_DIR)